# Step 1b (Option B) — Institution Labeling Pilot

**Goal of this notebook:** rebuild institution labels the Option B way.
We do NOT re-label the whole corpus. Instead we label two small sets with one
identical method, so fairness metrics are clean:
1. a random baseline sample of the corpus (unbiased corpus elite share)
2. (later) the retrieved set from Step 5a

**This notebook is the PILOT.** It runs on only 300 random papers first, just to
measure how much affiliation coverage the method gets before we commit to the
full run.

**Method:** OpenAlex lookup by arXiv id (landing_page_url), with a DOI fallback
(`10.48550/arXiv.<id>`) for papers not found the first way. Elite = affiliation
matches the QS World University Rankings 2026 Top 50 list.

**Cell 1 does:** imports, config (paths, email), and loads the two input files
(corpus + elite list). It prints the real Kaggle paths so we can confirm them.

In [1]:
# Step 1b Option B - PILOT - Cell 1: setup and load inputs
# Runs on Kaggle. Reads two Kaggle datasets:
#   - corpus:      fairsearch-qbio-processed-Raj-Jici-YB  (qbio_papers.json)
#   - elite list:  fairsearch-qbio-elite-list             (qs_top50_elite_2026.json)

import json
import os
import glob
import random

# --- Config ---
from kaggle_secrets import UserSecretsClient
OPENALEX_API_KEY = UserSecretsClient().get_secret("OPENALEX_API_KEY")  # from Kaggle Secrets
SAMPLE_N = 1000                          # full baseline sample size
RANDOM_SEED = 42                         # fixed for reproducibility

# --- Expected Kaggle paths (may differ; auto-detect below if not found) ---
CORPUS_PATH_EXPECTED = "/kaggle/input/fairsearch-qbio-processed-Raj-Jici-YB/qbio_papers.json"
ELITE_PATH_EXPECTED  = "/kaggle/input/fairsearch-qbio-elite-list/qs_top50_elite_2026.json"

def find_file(expected_path, filename_hint):
    """Return expected_path if it exists, else search /kaggle/input for the file."""
    if os.path.exists(expected_path):
        return expected_path
    hits = glob.glob(f"/kaggle/input/**/{filename_hint}", recursive=True)
    if hits:
        print(f"[auto-detect] expected not found, using: {hits[0]}")
        return hits[0]
    # last resort: list everything so we can see what's actually mounted
    print(f"[ERROR] could not find {filename_hint}. Files under /kaggle/input:")
    for p in glob.glob("/kaggle/input/**/*", recursive=True):
        print("   ", p)
    raise FileNotFoundError(filename_hint)

CORPUS_PATH = find_file(CORPUS_PATH_EXPECTED, "qbio_papers.json")
ELITE_PATH  = find_file(ELITE_PATH_EXPECTED,  "qs_top50_elite_2026.json")

print("CORPUS_PATH =", CORPUS_PATH)
print("ELITE_PATH  =", ELITE_PATH)

# --- Load ---
with open(CORPUS_PATH) as f:
    corpus = json.load(f)
with open(ELITE_PATH) as f:
    elite_list = json.load(f)

print("corpus papers:", len(corpus))
print("elite institutions:", len(elite_list))
print("corpus sample record:", corpus[0])
print("elite sample record:", elite_list[0])

[auto-detect] expected not found, using: /kaggle/input/datasets/yanbochen928/fairsearch-qbio-processed-raj-jici-yb/qbio_papers.json
[auto-detect] expected not found, using: /kaggle/input/datasets/yanbochen928/qs-top50-elite-2026/qs_top50_elite_2026.json
CORPUS_PATH = /kaggle/input/datasets/yanbochen928/fairsearch-qbio-processed-raj-jici-yb/qbio_papers.json
ELITE_PATH  = /kaggle/input/datasets/yanbochen928/qs-top50-elite-2026/qs_top50_elite_2026.json
corpus papers: 55301
elite institutions: 50
corpus sample record: {'paper_id': '0704.0021', 'title': 'Molecular Synchronization Waves in Arrays of Allosterically Regulated Enzymes', 'abstract': 'Spatiotemporal pattern formation in a product-activated enzymic reaction at high enzyme concentrations is investigated. Stochastic simulations show that catalytic turnover cycles of individual enzymes can become coherent and that complex wave patterns of molecular synchronization can develop. The analysis based on the mean-field approximation indica

## Cell 2 — Draw the random pilot sample (300 papers)

We draw 300 papers at random from the corpus, with a fixed seed so the sample is
reproducible. We also classify each paper_id as "new format" (e.g. 1610.07213)
or "old format" (e.g. q-bio/0404009), because the DOI fallback may behave
differently for the two. This lets us read coverage separately for each format
later.

This cell does NOT call OpenAlex yet. It only selects the sample and shows the
new/old split.

In [2]:
# Step 1b Option B - PILOT - Cell 2: draw random 300-paper sample + id format split

# Reproducible random sample
random.seed(RANDOM_SEED)
sample = random.sample(corpus, SAMPLE_N)
print(f"drew {len(sample)} papers (seed={RANDOM_SEED})")

def id_format(paper_id):
    """Classify arXiv id format.
    'old' = has a category prefix with a slash, e.g. q-bio/0404009
    'new' = numeric YYMM.number, e.g. 1610.07213
    """
    return "old" if "/" in paper_id else "new"

# Count the new/old split in the sample
split = {"new": 0, "old": 0}
for p in sample:
    split[id_format(p["paper_id"])] += 1

print("id format split in sample:")
print("   new format (e.g. 1610.07213):", split["new"])
print("   old format (e.g. q-bio/0404009):", split["old"])

# Show a couple of example ids of each kind so we can eyeball them
examples_new = [p["paper_id"] for p in sample if id_format(p["paper_id"]) == "new"][:3]
examples_old = [p["paper_id"] for p in sample if id_format(p["paper_id"]) == "old"][:3]
print("   new examples:", examples_new)
print("   old examples:", examples_old)

drew 1000 papers (seed=42)
id format split in sample:
   new format (e.g. 1610.07213): 921
   old format (e.g. q-bio/0404009): 79
   new examples: ['2409.04321', '1301.7745', '0812.3898']
   old examples: ['physics/0310009', 'physics/0110075', 'physics/0208040']


## Cell 3 — Batched OpenAlex lookup (50 ids per request)

Rewrites the per-paper lookup as a batch lookup. Instead of one request per
paper (which hit 429s and was slow), we ask about up to 50 papers in a single
request using OpenAlex's pipe (|) OR syntax on landing_page_url. This is the
method OpenAlex officially recommends and drops ~1000 papers from ~2000 requests
to ~20.

How results map back to each arXiv id: OpenAlex does NOT expose ids.arxiv, but
every arXiv work lists http://arxiv.org/abs/<id> in its locations landing_page_url.
So we parse that url out of each returned work to know which paper it is.

For each returned work we read the first author's institution. Three states,
same as before: found / no_affiliation. Any id we asked for but did NOT get back
is left for the DOI fallback (handled in Cell 4).

This cell defines the batch function and tests it on 5 ids.

In [3]:
# Step 1b Option B - FULL - Cell 3: batched OpenAlex lookup (50 ids/request)

import requests
import time
import re

OPENALEX_WORKS = "https://api.openalex.org/works"
BATCH_SIZE = 50   # OpenAlex OR-syntax max values per filter

def _first_institution(work):
    """(institution_display_name, country_code) from a work, or (None, None)."""
    for authorship in work.get("authorships", []):
        for inst in authorship.get("institutions", []):
            name = inst.get("display_name")
            country = inst.get("country_code")
            if name:
                return name, country
    return None, None

def _arxiv_id_from_work(work):
    """Pull the arXiv id out of a work's landing_page_urls.
    Returns the id string (e.g. '1610.07213' or 'physics/0310009') or None."""
    for loc in work.get("locations", []):
        url = loc.get("landing_page_url") or ""
        m = re.search(r"arxiv\.org/abs/(.+)$", url)
        if m:
            return m.group(1)
    return None

def lookup_batch_landing(id_list):
    """Query up to 50 ids at once by landing_page_url.
    Returns dict: arxiv_id -> {coverage, institution, country, matched_by}
    Only ids that OpenAlex returned are in the dict; missing ids are the caller's
    responsibility (they go to DOI fallback)."""
    out = {}
    urls = [f"http://arxiv.org/abs/{pid}" for pid in id_list]
    or_filter = "locations.landing_page_url:" + "|".join(urls)
    try:
        r = requests.get(
            OPENALEX_WORKS,
            params={"api_key": OPENALEX_API_KEY, "filter": or_filter,
                    "per-page": BATCH_SIZE},
            timeout=60,
        )
        r.raise_for_status()
        results = r.json().get("results", [])
    except Exception as e:
        print(f"   [warn] batch landing lookup error: {e}")
        results = []

    for w in results:
        aid = _arxiv_id_from_work(w)
        if aid is None:
            continue
        name, country = _first_institution(w)
        if name is None:
            out[aid] = {"coverage": "no_affiliation", "institution": None,
                        "country": None, "matched_by": "landing_page_url"}
        else:
            out[aid] = {"coverage": "found", "institution": name,
                        "country": country, "matched_by": "landing_page_url"}
    return out

# --- Test on 5 ids (new + old format) ---
test_ids = ["2409.04321", "0812.3898", "physics/0310009", "1301.7745", "1610.07213"]
test_out = lookup_batch_landing(test_ids)
print("asked:", len(test_ids), " returned:", len(test_out))
for pid in test_ids:
    print(pid, "->", test_out.get(pid, "NOT RETURNED (would go to DOI fallback)"))

asked: 5  returned: 5
2409.04321 -> {'coverage': 'no_affiliation', 'institution': None, 'country': None, 'matched_by': 'landing_page_url'}
0812.3898 -> {'coverage': 'found', 'institution': 'Foundation for Research and Technology Hellas', 'country': 'GR', 'matched_by': 'landing_page_url'}
physics/0310009 -> {'coverage': 'no_affiliation', 'institution': None, 'country': None, 'matched_by': 'landing_page_url'}
1301.7745 -> {'coverage': 'found', 'institution': 'Cold Spring Harbor Laboratory', 'country': 'US', 'matched_by': 'landing_page_url'}
1610.07213 -> {'coverage': 'no_affiliation', 'institution': None, 'country': None, 'matched_by': 'landing_page_url'}


## Cell 3.2 — Diagnostic: batched lookup, how to map results back to arXiv id

Before rewriting Cell 3/4 as batches, confirm the one new thing batching adds:
when we ask for 50 papers in one request, the response is a mixed list, so we
need a reliable key to map each returned work back to the arXiv id we asked for.

This sends 5 ids (new + old format) in a single request using the pipe (|) OR
syntax on landing_page_url, then prints, for each returned work, the fields that
could serve as that key. Sends only ONE request. Throwaway diagnostic.

In [4]:
# Step 1b Option B - FULL - Cell 3.2: diagnostic for batched result -> arxiv id mapping

diag_ids = ["2409.04321", "0812.3898", "physics/0310009", "1301.7745", "1610.07213"]

# Build ONE landing_page_url OR filter for all 5 ids (pipe-separated)
urls = [f"http://arxiv.org/abs/{pid}" for pid in diag_ids]
or_filter = "locations.landing_page_url:" + "|".join(urls)

r = requests.get(
    OPENALEX_WORKS,
    params={"api_key": OPENALEX_API_KEY, "filter": or_filter, "per-page": 50},
    timeout=30,
)
print("status:", r.status_code)
r.raise_for_status()
data = r.json()
print("meta count:", data.get("meta", {}).get("count"))
print("results returned:", len(data.get("results", [])))

# For each returned work, print fields that might key back to our arxiv id
for w in data.get("results", []):
    ids = w.get("ids", {})
    lps = [loc.get("landing_page_url") for loc in w.get("locations", [])]
    print("\n---")
    print("openalex id :", w.get("id"))
    print("ids.doi     :", ids.get("doi"))
    print("ids.arxiv   :", ids.get("arxiv"))
    print("landing_page_urls:")
    for u in lps:
        print("    ", repr(u))

status: 200
meta count: 5
results returned: 5

---
openalex id : https://openalex.org/W2149350210
ids.doi     : https://doi.org/10.1073/pnas.1309933111
ids.arxiv   : None
landing_page_urls:
     'https://doi.org/10.1073/pnas.1309933111'
     'https://pubmed.ncbi.nlm.nih.gov/24550517'
     None
     'http://arxiv.org/abs/1301.7745'
     'http://citeseerx.ist.psu.edu/viewdoc/summary?doi=10.1.1.768.3343'
     'http://europepmc.org/articles/PMC3948249'
     'https://www.ncbi.nlm.nih.gov/pmc/articles/3948249'

---
openalex id : https://openalex.org/W1697270980
ids.doi     : https://doi.org/10.1080/14786430802603746
ids.arxiv   : None
landing_page_urls:
     'https://doi.org/10.1080/14786430802603746'
     'http://arxiv.org/abs/0812.3898'
     'https://hal.science/hal-00513999'
     'http://hdl.handle.net/2262/47679'

---
openalex id : https://openalex.org/W2541318346
ids.doi     : https://doi.org/10.48550/arxiv.1610.07213
ids.arxiv   : None
landing_page_urls:
     'http://arxiv.org/abs/1610

## Cell 4 — Run batched lookup on all sampled papers (primary + DOI fallback)

Two rounds, both batched at 50 ids/request:
1. Primary: landing_page_url batches (uses lookup_batch_landing from Cell 3).
   Collect any ids NOT returned by this round.
2. DOI fallback: for the leftover ids, query in batches by DOI
   (10.48550/arXiv.<id>). Same 3-state logic.

Anything still not returned after both rounds is not_found. Final result per
paper: paper_id, id_format, coverage, institution, country, matched_by.
This replaces ~2000 single requests with ~40 batch requests, so it finishes in
seconds and stays well inside the free daily budget.

In [5]:
# Step 1b Option B - FULL - Cell 4: batched run over the sample (primary + DOI fallback)

def _doi_from_work(work):
    """Return the work's DOI url (lowercased) or ''."""
    return (work.get("ids", {}).get("doi") or "").lower()

def lookup_batch_doi(id_list):
    """Query up to 50 ids at once by arXiv DOI (10.48550/arXiv.<id>).
    Returns dict: arxiv_id -> {coverage, institution, country, matched_by}."""
    out = {}
    doi_by_id = {pid: f"10.48550/arxiv.{pid}".lower() for pid in id_list}
    dois = list(doi_by_id.values())
    or_filter = "doi:" + "|".join(dois)
    try:
        r = requests.get(
            OPENALEX_WORKS,
            params={"api_key": OPENALEX_API_KEY, "filter": or_filter, "per-page": BATCH_SIZE},
            timeout=60,
        )
        r.raise_for_status()
        results = r.json().get("results", [])
    except Exception as e:
        print(f"   [warn] batch doi lookup error: {e}")
        results = []

    id_by_doi = {}
    for pid, doi in doi_by_id.items():
        id_by_doi[f"https://doi.org/{doi}"] = pid

    for w in results:
        wdoi = _doi_from_work(w)
        pid = id_by_doi.get(wdoi)
        if pid is None:
            continue
        name, country = _first_institution(w)
        if name is None:
            out[pid] = {"coverage": "no_affiliation", "institution": None, "country": None, "matched_by": "doi_fallback"}
        else:
            out[pid] = {"coverage": "found", "institution": name, "country": country, "matched_by": "doi_fallback"}
    return out

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

all_ids = [p["paper_id"] for p in sample]
t_start = time.time()

# Round 1: primary landing_page_url batches
result_map = {}
for batch in chunked(all_ids, BATCH_SIZE):
    result_map.update(lookup_batch_landing(batch))
primary_n = len(result_map)
primary_t = time.time() - t_start
print(f"after primary: {primary_n}/{len(all_ids)} resolved ({primary_t:.0f}s)")

# Round 2: DOI fallback for ids not resolved in round 1
missing = [pid for pid in all_ids if pid not in result_map]
for batch in chunked(missing, BATCH_SIZE):
    result_map.update(lookup_batch_doi(batch))
fallback_n = len(result_map)
fallback_t = time.time() - t_start
print(f"after doi fallback: {fallback_n}/{len(all_ids)} resolved ({fallback_t:.0f}s)")

# Assemble final results (mark leftovers as not_found)
results_full = []
for p in sample:
    pid = p["paper_id"]
    rec = result_map.get(pid, {"coverage": "not_found", "institution": None, "country": None, "matched_by": None})
    rec = dict(rec)
    rec["paper_id"] = pid
    rec["id_format"] = id_format(pid)
    results_full.append(rec)

print(f"\nfinished {len(results_full)} papers in {time.time()-t_start:.0f}s")
from collections import Counter
cov = Counter(r["coverage"] for r in results_full)
print("coverage tally:", dict(cov))

after primary: 919/1000 resolved (25s)
after doi fallback: 979/1000 resolved (28s)

finished 1000 papers in 28s
coverage tally: {'no_affiliation': 541, 'found': 438, 'not_found': 21}


## Cell 5 — Coverage by id format + elite share of the sample

1. Break coverage down by id format (new vs old).
2. Apply the QS Top-50 elite list to the `found` papers and compute the elite
   share of this random sample. This sample elite share is the UNBIASED BASELINE
   that Option B's SPD uses (SPD = retrieved_share - sample_share).

Matching: exact, case-insensitive match of the OpenAlex institution display name
against the QS Top-50 names.

In [6]:
# Step 1b Option B - FULL - Cell 5: coverage by id format + elite share

from collections import defaultdict
by_fmt = defaultdict(Counter)
for r in results_full:
    by_fmt[r["id_format"]][r["coverage"]] += 1

print("coverage by id format:")
for fmt in ["new", "old"]:
    c = by_fmt[fmt]
    total = sum(c.values())
    found = c.get("found", 0)
    if total:
        print(f"   {fmt}: total={total}, found={found} ({found/total:.0%}), no_affiliation={c.get('no_affiliation',0)}, not_found={c.get('not_found',0)}")

def norm(name):
    return name.strip().lower() if name else ""

elite_names = {norm(e["name"]) for e in elite_list}

found_recs = [r for r in results_full if r["coverage"] == "found"]
elite_count = 0
for r in found_recs:
    r["elite_label"] = 1 if norm(r["institution"]) in elite_names else 0
    elite_count += r["elite_label"]

n_found = len(found_recs)
print("\nelite share of the random sample (baseline):")
print(f"   found papers        : {n_found}")
print(f"   elite (QS Top-50)   : {elite_count}")
if n_found:
    print(f"   sample elite share  : {elite_count/n_found:.3f}")

elite_hits = Counter(r["institution"] for r in found_recs if r["elite_label"] == 1)
print("\nelite institutions matched in sample:")
for name, cnt in elite_hits.most_common():
    print(f"   {cnt}x  {name}")

coverage by id format:
   new: total=921, found=379 (41%), no_affiliation=521, not_found=21
   old: total=79, found=59 (75%), no_affiliation=20, not_found=0

elite share of the random sample (baseline):
   found papers        : 438
   elite (QS Top-50)   : 63
   sample elite share  : 0.144

elite institutions matched in sample:
   7x  Princeton University
   6x  University of Oxford
   5x  University of Pennsylvania
   4x  Massachusetts Institute of Technology
   4x  Harvard University
   3x  The University of Queensland
   3x  University of Cambridge
   3x  University of California, Los Angeles
   3x  University of British Columbia
   3x  Cornell University
   2x  University of Chicago
   2x  Yale University
   2x  Imperial College London
   2x  ETH Zurich
   2x  University of California, Berkeley
   2x  Stanford University
   2x  Northwestern University
   2x  Johns Hopkins University
   1x  Technical University of Munich
   1x  Seoul National University
   1x  The University of Toky

## Cell 6 — Save the 1000-paper sample labels

Write results_full to JSON in /kaggle/working/ so it can be downloaded and later
used as the Option B baseline input for Step 5b. Also print a short summary so
the file's provenance is self-documenting.

In [7]:
# Step 1b Option B - FULL - Cell 6: save the 1000-paper sample labels

import json
from collections import Counter

OUT_PATH = "/kaggle/working/sample_labels_1000.json"

cov = Counter(r["coverage"] for r in results_full)
found_recs = [r for r in results_full if r["coverage"] == "found"]
elite_count = sum(r.get("elite_label", 0) for r in found_recs)
n_found = len(found_recs)

payload = {
    "meta": {
        "role": "option_b_baseline_sample",
        "sample_n": SAMPLE_N,
        "random_seed": RANDOM_SEED,
        "elite_list": "QS World University Rankings 2026 Top 50",
        "method": "OpenAlex batched (landing_page_url primary + DOI fallback), api_key",
        "coverage": dict(cov),
        "found": n_found,
        "elite_found": elite_count,
        "sample_elite_share": round(elite_count / n_found, 4) if n_found else None,
    },
    "labels": results_full,
}

with open(OUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print("saved:", OUT_PATH)
print("meta:", json.dumps(payload["meta"], indent=2))

saved: /kaggle/working/sample_labels_1000.json
meta: {
  "role": "option_b_baseline_sample",
  "sample_n": 1000,
  "random_seed": 42,
  "elite_list": "QS World University Rankings 2026 Top 50",
  "method": "OpenAlex batched (landing_page_url primary + DOI fallback), api_key",
  "coverage": {
    "no_affiliation": 541,
    "found": 438,
    "not_found": 21
  },
  "found": 438,
  "elite_found": 63,
  "sample_elite_share": 0.1438
}
